# ⚙️ Módulos y Librerías: Operaciones de Bajo Nivel en `os` (`os.open` y `os.close`)

| Propiedad | Detalle |
| :--- | :--- |
| 📍 **Módulo** | Módulos, Paquetes y Librerías / Módulo OS |
| 🎯 **Objetivo** | Comprender la diferencia entre la función de alto nivel `open()` de Python y la llamada del módulo de bajo nivel `os.open()`, gestionando descriptores de archivos (*file descriptors*) enteramente a través de primitivas del sistema operativo. |
| 🛠️ **Conceptos** | `os.open()`, `os.close()`, `File Descriptors (int)`, `Flags de Control (os.O_RDONLY, os.O_WRONLY, os.O_CREAT)`, `Permisos POSIX` |

---

### 💡 Apertura de Bajo Nivel con `os.open()` y `os.close()`

A diferencia de la función integrada `open()` que retorna un objeto de alto nivel (*file stream/handle*), **`os.open()`** realiza una llamada directa a la API subyacente del sistema operativo (System Call) y retorna un **descriptor de archivo** (*file descriptor*), representado como un número entero (`int`).

---

### 🗂️ Banderas (*Flags*) de Control Comunes en `os.open()`

Para especificar el modo de acceso en `os.open()`, se utilizan constantes o banderas accesibles desde el módulo `os`:

| Flag | Descripción de la Operación |
| :--- | :--- |
| 📖 **`os.O_RDONLY`** | Abre el archivo únicamente en modo lectura. |
| ✍️ **`os.O_WRONLY`** | Abre el archivo únicamente en modo escritura. |
| 🔄 **`os.O_RDWR`** | Abre el archivo en modo lectura y escritura. |
| ➕ **`os.O_CREAT`** | Crea el archivo si no existe previamente en el sistema. |
| ✂️ **`os.O_TRUNC`** | Trunca (vacía) el archivo a cero bytes si existe al abrirlo. |

In [ ]:
import os
from typing import Any

# 🛰️ [1] Banner visual de inicio de ejecución
print("=" * 80)
print("🐍 MÓDULO OS | Apertura y Cierre de Bajo Nivel (os.open & os.close)")
print("=" * 80)

# ================================================================================
# 🧪 1. APERTURA Y CIERRE BÁSICO CON DESCRIPTORES DE ARCHIVO
# ================================================================================
# Manipulación directa de file descriptors enteros con llamadas del SO.
# ================================================================================

print("\n--- 🚀 [PASO 1]: Uso de os.open() y os.close() ---")

sample_file_name: str = "low_level_file.txt"

# Creación inicial de un archivo para la prueba
with open(sample_file_name, "w", encoding="utf-8") as setup_file:
    setup_file.write("Contenido de prueba para descriptores de bajo nivel.")

file_descriptor: int | None = None

try:
    # 🔌 [1] Abre el archivo en bajo nivel (Modo Solo Lectura)
    file_descriptor = os.open(sample_file_name, os.O_RDONLY)

    print(f"📋 Descriptor de archivo obtenido (int): {file_descriptor}")
    print(f"  🔹 Tipo de dato retornado: {type(file_descriptor)}")

finally:
    # 🔌 [2] Cierra el archivo usando su descriptor numérico
    if file_descriptor is not None:
        os.close(file_descriptor)
        print("  ✅ Descriptor de archivo cerrado correctamente mediante os.close().")

    # Limpieza del archivo de demostración
    if os.path.exists(sample_file_name):
        os.remove(sample_file_name)

## 🧱 Creación y Escritura Directa usando Banderas Combinadas

### 💡 Máscaras de Banderas (*Bitwise OR*):
En operaciones de bajo nivel, las banderas se combinan mediante el operador de bit `|` (OR a nivel de bits). Cuando se utiliza la bandera `os.O_CREAT`, es obligatorio o altamente recomendado especificar el parámetro de permisos POSIX (por ejemplo, `0o644` para lectura/escritura del propietario).

In [ ]:
# ================================================================================
# 🧪 2. CREACIÓN CON BANDERAS Y PERMISOS POSIX
# ================================================================================
print("\n--- 🚀 [PASO 2]: Creación y Control de Permisos con os.open() ---")

new_low_level_file: str = "created_with_flags.txt"

# ⚙️ [1] Definición de banderas combinadas y modo de permisos
creation_flags: int = os.O_WRONLY | os.O_CREAT | os.O_TRUNC
file_mode: int = 0o644  # Lectura/Escritura para el propietario, Lectura para grupo/otros

writer_fd: int | None = None

try:
    # ⚙️ [2] Apertura/Creación con descriptor de bajo nivel
    writer_fd = os.open(new_low_level_file, creation_flags, file_mode)
    print(f"📋 Descriptor de escritura instanciado: {writer_fd}")

    # Escribir datos mediante os.write() recibiendo bytes
    message_bytes: bytes = "Escritura mediante primitivas del SO.\n".encode("utf-8")
    bytes_written: int = os.write(writer_fd, message_bytes)
    print(f"  🔹 Bytes escritos directamente en disco: {bytes_written}")

finally:
    if writer_fd is not None:
        os.close(writer_fd)
        print("  ✅ Descriptor de escritura liberado.")

    if os.path.exists(new_low_level_file):
        os.remove(new_low_level_file)

## 🛒 Ejercicios Prácticos Avanzados Integrados

---

### 🧪 Ejercicio 1: Administrador de Descriptores Seguros para Operaciones de Sistema

**Escenario Empresarial:** Se necesita implementar un gestor que intente abrir un recurso crítico en bajo nivel, realice comprobaciones sobre el entero del *file descriptor* asignado y asegure que `os.close()` sea invocado bajo cualquier circunstancia, evitando la fuga de descriptores a nivel de sistema operativo.

In [ ]:
# ================================================================================
# 🧪 3. EJERCICIO PRÁCTICO 1: GESTOR RESILIENTE DE FILE DESCRIPTORS
# ================================================================================
print("\n--- 🧪 [PASO 3.1]: Ejercicio 1 - Gestor de Descriptores OS ---")


def execute_low_level_file_inspection(file_path: str) -> dict[str, Any]:
    """Abre un archivo utilizando os.open, inspecciona el descriptor y asegura el cierre.

    Args:
        file_path (str): Ruta del archivo a inspeccionar en bajo nivel.

    Returns:
        dict[str, Any]: Reporte del descriptor e historial de ejecución.
    """
    inspection_report: dict[str, Any] = {
        "success": False,
        "file_descriptor": None,
        "error": None
    }

    target_fd: int | None = None

    try:
        # [1] Intentar la apertura en bajo nivel
        target_fd = os.open(file_path, os.O_RDONLY)
        inspection_report["success"] = True
        inspection_report["file_descriptor"] = target_fd
        print(f"  ✅ Recurso abierto. Descriptor asignado por el kernel: {target_fd}")

    except OSError as os_err:
        inspection_report["error"] = f"Error del SO al abrir el archivo: {os_err}"
        print(f"  ❌ {inspection_report['error']}")

    finally:
        # [2] Garantizar liberación del descriptor
        if target_fd is not None:
            os.close(target_fd)
            print(f"  🔒 Descriptor {target_fd} liberado exitosamente mediante os.close().")

    return inspection_report


# Prueba del ejercicio
audit_file: str = "system_audit.tmp"

# Preparación de archivo de prueba
with open(audit_file, "w", encoding="utf-8") as temp_file:
    temp_file.write("Registro de auditoría del kernel.\n")

print(f"📋 Ejecutando inspección de bajo nivel sobre '{audit_file}'...")
report: dict[str, Any] = execute_low_level_file_inspection(audit_file)

print("\n📋 Resumen del Diagnóstico:")
print(f"  🔹 Estado de ejecución: {report['success']}")
print(f"  🔹 Descriptor asignado: {report['file_descriptor']}")

# Limpieza del archivo de prueba
if os.path.exists(audit_file):
    os.remove(audit_file)

# --------------------------------------------------------------------------------
# 🏁 [PASO 4]: Cierre de Ejecución
# --------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("✅ [FIN DE EJECUCIÓN] Lección de os.open() y os.close() finalizada.")
print("=" * 80)